# **Medical Agent - Question-Answering System with LangChain and Pinecone**

## Overview

This notebook implements a medical question-answering system that processes PDF documents from the *Gale Encyclopedia of Medicine (3rd Edition)*. The system uses LangChain for document loading, text splitting, and retrieval-augmented generation (RAG), Pinecone for vector storage, and Google's Generative AI for embeddings and chat responses. Users can query medical topics (e.g., "What is Acne? and how is it treated?"), and the system retrieves relevant information from the processed documents to provide accurate, empathetic responses.

### Pipeline Steps

1. **Load PDFs**: Extract text from PDF files in a specified directory.
2. **Preprocess Documents**: Filter metadata and split documents into smaller chunks.
3. **Generate Embeddings**: Convert text chunks into vector embeddings using Google Generative AI.
4. **Store in Pinecone**: Store embeddings in a Pinecone vector store for efficient retrieval.
5. **Build Retrieval Chain**: Combine a retriever and a chat model to answer queries using retrieved documents.
6. **Query Processing**: Respond to user queries with information solely from the retrieved documents.


## Prerequisites

Before running this notebook, ensure the following:

### Required Libraries

Install the necessary Python libraries using pip:

```bash
pip install langchain langchain-community langchain-google-genai pinecone-client langchain-pinecone
```

### Environment Variables

Set the following environment variables:

- `GEMINI_API_KEY`: Your Google Generative AI API key.
- `PINECONE_API_KEY`: Your Pinecone API key.

You can set these in your environment or in the notebook using:

```python
import os
os.environ["GEMINI_API_KEY"] = "your-gemini-api-key"
os.environ["PINECONE_API_KEY"] = "your-pinecone-api-key"
```

### Data

Place your PDF files (e.g., from the *Gale Encyclopedia of Medicine*) in a directory named `data` in the same directory as this notebook.

## Setup

### Import Libraries

In [2]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from typing import List
from langchain.schema import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain
import os

### Verify Environment Variables

In [4]:
%pwd

'd:\\AI Personal Projects\\GenAI - AI Agents\\Medical-Agent\\research'

In [5]:
os.chdir('../')

In [6]:
%pwd

'd:\\AI Personal Projects\\GenAI - AI Agents\\Medical-Agent'

In [7]:
if not os.getenv("GEMINI_API_KEY") or not os.getenv("PINECONE_API_KEY"):
    raise ValueError("Please set GEMINI_API_KEY and PINECONE_API_KEY environment variables.")

## Document Loading and Preprocessing

### Load PDFs

Load all PDF files from the `data` directory.

In [10]:
def load_pdfs_from_directory(directory: str) -> List[Document]:
    """Load all PDF documents from a specified directory.

    Args:
        directory (str): Path to the directory containing PDF files.

    Returns:
        List[Document]: A list of Document objects loaded from the PDF files.

    Notes:
        Uses DirectoryLoader with PyPDFLoader to recursively load all PDFs
        in the specified directory and its subdirectories.
    """
    loader = DirectoryLoader(directory, glob="**/*.pdf", loader_cls=PyPDFLoader)
    documents = loader.load()
    return documents

### Filter Documents

Reduce metadata to only the `source` field to optimize memory usage.

In [11]:
def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """Filter documents to retain only essential metadata (source).

    Args:
        docs (List[Document]): List of Document objects to filter.

    Returns:
        List[Document]: A new list of Document objects with only the 'source'
        metadata field retained.

    Notes:
        This function creates new Document objects with minimal metadata to
        reduce memory usage while preserving the source information.
    """
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get('source')
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={'source': src}
            )
        )
    return minimal_docs

### Split Documents

In [12]:
def text_splitter(documents: List[Document], chunk_size: int = 500, 
                  chunk_overlap: int = 20) -> List[Document]:
    """Split documents into smaller chunks for processing.

    Args:
        documents (List[Document]): List of Document objects to split.
        chunk_size (int, optional): Maximum size of each chunk in characters.
            Defaults to 500.
        chunk_overlap (int, optional): Number of characters to overlap between
            chunks. Defaults to 20.

    Returns:
        List[Document]: A list of Document objects representing the split chunks.

    Notes:
        Uses RecursiveCharacterTextSplitter to split documents into smaller
        chunks suitable for embedding and retrieval.
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    text_chunks = text_splitter.split_documents(documents)
    return text_chunks

## Embedding and Vector Storage

### Initialize Embeddings

Set up the Google Generative AI embeddings model.

In [13]:
def get_embeddings() -> GoogleGenerativeAIEmbeddings:
    """Initialize and return Google Generative AI embeddings model.

    Returns:
        GoogleGenerativeAIEmbeddings: Configured embeddings model instance.

    Notes:
        Requires the GEMINI_API_KEY environment variable to be set.
        The model used is 'models/embedding-001' with a timeout of 10 seconds.
    """
    embeddings = GoogleGenerativeAIEmbeddings(
        model="models/embedding-001",
        google_api_key=os.getenv("GEMINI_API_KEY"),
        request_options={"timeout": 10}
    )
    return embeddings

### Create Pinecone Index

Set up a Pinecone index for storing document embeddings.

In [14]:
def create_pinecone_index(index_name: str) -> Pinecone.Index:
    """Create or connect to a Pinecone index for vector storage.

    Args:
        index_name (str): Name of the Pinecone index to create or connect to.

    Returns:
        Pinecone.Index: The Pinecone index object.

    Notes:
        Requires the PINECONE_API_KEY environment variable to be set.
        Creates a serverless index with dimension 768, cosine metric, and AWS
        us-east-1 region if it does not already exist.
    """
    pinecone = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
    if not pinecone.has_index(index_name):
        pinecone.create_index(
            name=index_name,
            dimension=768,
            metric="cosine",
            spec=ServerlessSpec(cloud="aws", region="us-east-1")
        )
    pinecone_index = pinecone.Index(index_name)
    return pinecone_index

### Store Embeddings

Store the document chunks as embeddings in Pinecone.

In [15]:
def store_embeddings(index_name: str, text_chunks: List[Document], 
                     embeddings: GoogleGenerativeAIEmbeddings) -> PineconeVectorStore:
    """Store document embeddings in a Pinecone vector store.

    Args:
        index_name (str): Name of the Pinecone index to store embeddings in.
        text_chunks (List[Document]): List of document chunks to embed.
        embeddings (GoogleGenerativeAIEmbeddings): Embeddings model to use.

    Returns:
        PineconeVectorStore: The vector store containing the embedded documents.
    """
    vector_store = PineconeVectorStore.from_documents(
        documents=text_chunks,
        index_name=index_name,
        embedding=embeddings
    )
    return vector_store

### Load Existing Vector Store

Connect to an existing Pinecone vector store for retrieval.

In [16]:
def load_existing_vector_store(index_name: str, 
                               embeddings: GoogleGenerativeAIEmbeddings) -> PineconeVectorStore:
    """Load an existing Pinecone vector store.

    Args:
        index_name (str): Name of the existing Pinecone index to load.
        embeddings (GoogleGenerativeAIEmbeddings): Embeddings model to use.

    Returns:
        PineconeVectorStore: The loaded vector store.
    """
    vector_store = PineconeVectorStore.from_existing_index(
        index_name=index_name,
        embedding=embeddings
    )
    return vector_store

## Retrieval and Response Generation

### Create Retriever

Set up a retriever to fetch relevant documents from the vector store.

In [17]:
def retrieve_documents(vector_store: PineconeVectorStore):
    """Create a retriever from a Pinecone vector store.

    Args:
        vector_store (PineconeVectorStore): The vector store to create a
            retriever from.

    Returns:
        Retriever: A retriever configured for similarity search with k=5 results.

    Notes:
        Configures the retriever to return the top 5 most similar documents
        based on cosine similarity.
    """
    retriever = vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 5}
    )
    return retriever

### Initialize Chat Model

In [18]:
def get_chat_model() -> ChatGoogleGenerativeAI:
    """Initialize and return a Google Generative AI chat model.

    Returns:
        ChatGoogleGenerativeAI: Configured chat model instance.

    Notes:
        Requires the GEMINI_API_KEY environment variable to be set.
        Uses the 'models/gemini-2.5-flash' model.
    """
    chat_model = ChatGoogleGenerativeAI(
        model="models/gemini-2.5-flash",
        google_api_key=os.getenv("GEMINI_API_KEY")
    )
    return chat_model

### Create Medical Prompt

Define a prompt template for empathetic medical responses.

In [19]:
def get_medical_prompt() -> ChatPromptTemplate:
    """Create a medical expert prompt template for the chat model.

    Returns:
        ChatPromptTemplate: A prompt template configured for medical queries.

    Notes:
        The prompt is designed to provide empathetic, accurate responses based
        on retrieved documents from the Gale Encyclopedia of Medicine (3rd Edition).
        It includes constraints to ensure responses are based only on provided
        documents and avoid speculation.
    """
    system_prompt = (
        "You are a compassionate and professional medical assistant trained on the Gale Encyclopedia of Medicine (3rd Edition). "
        "Your role is to provide helpful, accurate, and clear responses **only using the retrieved documents**. "
        "You should address the user's concerns with empathy and explain medical terms in a way that's easy for a general audience to understand.\n\n"
        
        "If the user describes personal symptoms (e.g., 'I have a fever', 'I'm feeling dizzy', etc.), respond with a caring tone, acknowledge their symptoms, "
        "and explain potential causes or treatments based on the retrieved information. For example, you can say things like:\n"
        "- 'I'm sorry you're feeling that way.'\n"
        "- 'Let’s look into what might be causing your symptoms.'\n"
        "- 'Based on the encyclopedia, here's what you should know...'\n\n"
        
        "**Important constraints:**\n"
        "- Only use content from the retrieved documents.\n"
        "- Do not speculate or offer personal opinions.\n"
        "- Do not reference page numbers or unavailable information (e.g., 'The provided text only includes the title...').\n"
        "- If the documents do not provide enough information, reply: 'I'm sorry, I don't have enough information from the provided sources to answer that.'\n\n"
        
        "{context}"
    )

    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "{input}"),
    ])
    return prompt

### Build Document and Retrieval Chains

Combine the chat model, prompt, and retriever into a full pipeline.

In [22]:
def create_chain(llm: ChatGoogleGenerativeAI, prompt: ChatPromptTemplate):
    """Create a document chain for combining retrieved documents with the LLM.

    Args:
        llm (ChatGoogleGenerativeAI): The language model to use in the chain.
        prompt (ChatPromptTemplate): The prompt template to use for the chain.

    Returns:
        DocumentChain: A chain that combines documents and processes them with the LLM.

    Notes:
        Uses create_stuff_documents_chain to process retrieved documents with
        the provided language model and prompt.
    """
    chain = create_stuff_documents_chain(
        llm=llm,
        prompt=prompt
    )
    return chain

def build_retrieval_chain(retriever, chain):
    """Build a retrieval chain combining a retriever and document chain.

    Args:
        retriever: The retriever object to fetch relevant documents.
        chain: The document chain to process retrieved documents.

    Returns:
        RetrievalChain: A chain that retrieves and processes documents for query responses.

    Notes:
        Combines the retriever and document chain to create a full retrieval-augmented
        generation pipeline.
    """
    retrieval_chain = create_retrieval_chain(
        retriever=retriever,
        combine_docs_chain=chain
    )
    return retrieval_chain

### Pipeline

### Since I’ve already completed the following steps, I won’t be repeating them—you can handle this part on your own:
```python
extracted_docs = load_pdfs_from_directory("data")
minimal_docs = filter_to_minimal_docs(extracted_docs)
text_chunks = text_splitter(minimal_docs)
embeddings = get_embeddings()
pinecone_index = create_pinecone_index("medical-agent")
vector_store = store_embeddings(index_name="medical-agent", text_chunks=text_chunks, embeddings=embeddings)
```

In [23]:
embeddings = get_embeddings()
existing_vector_store = load_existing_vector_store(index_name="medical-agent", embeddings=embeddings)
retriever = retrieve_documents(existing_vector_store)
llm = get_chat_model()
prompt = get_medical_prompt()
chain = create_chain(llm, prompt)
retrieval_chain = build_retrieval_chain(retriever, chain)

## Example Queries

Test the pipeline with sample medical queries.

### Query 1: What is Acne and How is it Treated?

In [24]:
response = retrieval_chain.invoke({"input": "What is Acne? and how is it treated?"})
print("Query: What is Acne? and how is it treated?")
print("Answer:", response['answer'])

Query: What is Acne? and how is it treated?
Answer: Acne has a characteristic appearance and is not difficult to diagnose. It most commonly occurs on the face, chest, shoulders, and back, as these areas have the most sebaceous follicles (which are tiny sacs under the skin that produce oil).

Regarding treatment, the provided information states that acne tends to reappear when treatment stops, but it usually improves on its own over time. Inflammatory acne may leave scars that require further treatment. The document does not detail specific treatments for acne, but it does offer steps to minimize flare-ups:
*   Gently wash affected areas once or twice daily.
*   Avoid abrasive cleansers.
*   Use noncomedogenic (products that won't clog pores) makeup and moisturizers.
*   Shampoo often and keep hair off your face.
*   Eat a well-balanced diet, avoiding foods that trigger flare-ups for you.


### Query 2: I Have a Fever, What Should I Do?

In [25]:
response = retrieval_chain.invoke({"input": "I have a fever, what should I do?"})
print("Query: I have a fever, what should I do?")
print("Answer:", response['answer'])

Query: I have a fever, what should I do?
Answer: I'm sorry you're feeling that way. Let’s look into what might be causing your symptoms and what the encyclopedia suggests you should know about fevers.

Bathing in cool water can help alleviate a high fever.

It's important to be aware that a fever requires emergency treatment under the following circumstances:

*   For a newborn (three months or younger) with a fever over 100.5°F (38°C).
*   For an infant or child with a fever over 103°F (39.4°C).
*   If the fever is accompanied by a severe headache, neck stiffness, mental confusion, or severe swelling of the throat.
*   A very high fever in a small child can trigger seizures (febrile seizures), so it should be carefully monitored.

Fevers are primarily caused by viral or bacterial infections, such as pneumonia or influenza, but other conditions can also induce a fever. A fever is usually diagnosed using a thermometer. For adults and older children, temperature readings are usually take

## Notes

- **Error Handling**: Ensure environment variables are set and the `data` directory contains valid PDFs. Add try-except blocks if needed for robustness.
- **Performance**: The pipeline may take time to process large PDFs or store embeddings. Consider chunking large datasets or using a smaller `chunk_size` for faster processing.
- **Extensibility**: You can modify the `chunk_size`, `chunk_overlap`, or `search_kwargs` (e.g., `k`) to tune performance and relevance of retrieved documents.
- **Safety**: Responses are constrained to the provided documents to avoid speculation. Always consult a healthcare professional for personal medical advice.

## Conclusion

This notebook provides a fully functional pipeline for answering medical queries using a retrieval-augmented generation approach. You can extend it by adding more PDFs, tweaking the prompt, or integrating additional features like query logging or response caching.